# Machine Learning Metrics:

This notebook provides a clear, step-by-step comparison between **naive (loop-based)** and **vectorized (NumPy-based)** implementations for common classification metrics. For each metric, you'll find:

- A concise explanation of the metric and its formula
- A naive implementation (using Python loops)
- A vectorized implementation (using NumPy for efficiency)
- Example usage and output

You'll also learn about the performance and readability benefits of vectorized code, and see how to use these metrics on a sample dataset.

---

## 1. Setup: Import Libraries and Create Example Data

Before we start, let's import the necessary libraries and create a small example dataset for binary classification. We'll use this dataset to demonstrate each metric.


In [13]:
import numpy as np
from sklearn.metrics import roc_auc_score, log_loss

## 2. Accuracy

**Accuracy** measures the proportion of correct predictions out of all predictions. It is the most basic metric for classification tasks.

**Formula:**
$$\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Predictions}}$$

We'll implement accuracy in two ways:
- **Naive:** Using a Python loop
- **Vectorized:** Using NumPy for efficiency

### Naive Implementation: Accuracy

This function uses a loop to count the number of correct predictions.

In [14]:
def accuracy_naive(true_labels, predicted_labels):
    """Compute accuracy using a Python loop."""
    correct_count = 0
    for actual, predicted in zip(true_labels, predicted_labels):
        if actual == predicted:
            correct_count += 1
    return correct_count / len(true_labels)

### Vectorized Implementation: Accuracy

This function uses NumPy to compare arrays directly, making the code more concise and much faster for large datasets.

In [15]:
def accuracy_vectorized(true_labels, predicted_labels):
    """Compute accuracy using NumPy vectorization."""
    return np.mean(true_labels == predicted_labels)

## 3. Precision, Recall, and F1 Score

These metrics are especially important for imbalanced datasets:
- **Precision:** Of all positive predictions, how many were correct?
- **Recall:** Of all actual positives, how many did we find?
- **F1 Score:** Harmonic mean of precision and recall.

**Formulas:**
$$\text{Precision} = \frac{TP}{TP + FP}$$
$$\text{Recall} = \frac{TP}{TP + FN}$$
$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

We'll implement these metrics both naively and vectorized.

### Naive Implementation: Precision, Recall, F1

This function uses loops to count true positives, false positives, and false negatives, then computes the metrics.

In [16]:
def precision_recall_f1_naive(true_labels, predicted_labels):
    """Compute precision, recall, and F1 using Python loops."""
    true_positive = false_positive = false_negative = 0
    for actual, predicted in zip(true_labels, predicted_labels):
        if predicted == 1 and actual == 1:
            true_positive += 1
        elif predicted == 1 and actual == 0:
            false_positive += 1
        elif predicted == 0 and actual == 1:
            false_negative += 1
    precision = true_positive / (true_positive + false_positive + 1e-8)
    recall = true_positive / (true_positive + false_negative + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1

### Vectorized Implementation: Precision, Recall, F1

This function uses NumPy to efficiently count true positives, false positives, and false negatives.

In [17]:
def precision_recall_f1_vectorized(true_labels, predicted_labels):
    """Compute precision, recall, and F1 using NumPy vectorization."""
    true_positive = np.sum((true_labels == 1) & (predicted_labels == 1))
    false_positive = np.sum((true_labels == 0) & (predicted_labels == 1))
    false_negative = np.sum((true_labels == 1) & (predicted_labels == 0))
    precision = true_positive / (true_positive + false_positive + 1e-8)
    recall = true_positive / (true_positive + false_negative + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1

## 4. Confusion Matrix

A **confusion matrix** summarizes the counts of true positives, false positives, true negatives, and false negatives. It is a useful diagnostic tool for classification models.

**Layout:**
|            | Predicted Positive | Predicted Negative |
|------------|-------------------|-------------------|
| Actual Pos | True Positive     | False Negative    |
| Actual Neg | False Positive    | True Negative     |

We'll implement both naive and vectorized versions.

### Naive Implementation: Confusion Matrix

This function uses loops to count each cell of the confusion matrix.

In [18]:
def confusion_matrix_naive(true_labels, predicted_labels):
    """Compute confusion matrix using Python loops."""
    true_positive = false_positive = true_negative = false_negative = 0
    for actual, predicted in zip(true_labels, predicted_labels):
        if actual == 1 and predicted == 1:
            true_positive += 1
        elif actual == 0 and predicted == 1:
            false_positive += 1
        elif actual == 0 and predicted == 0:
            true_negative += 1
        elif actual == 1 and predicted == 0:
            false_negative += 1
    return np.array([[true_positive, false_positive], [false_negative, true_negative]])

### Vectorized Implementation: Confusion Matrix

This function uses NumPy to efficiently count each cell of the confusion matrix.

In [19]:
def confusion_matrix_vectorized(true_labels, predicted_labels):
    """Compute confusion matrix using NumPy vectorization."""
    true_positive = np.sum((true_labels == 1) & (predicted_labels == 1))
    false_positive = np.sum((true_labels == 0) & (predicted_labels == 1))
    false_negative = np.sum((true_labels == 1) & (predicted_labels == 0))
    true_negative = np.sum((true_labels == 0) & (predicted_labels == 0))
    return np.array([[true_positive, false_positive], [false_negative, true_negative]])

## 5. ROC-AUC (Receiver Operating Characteristic - Area Under Curve)

**ROC-AUC** measures the ability of a classifier to distinguish between classes, plotting the true positive rate (TPR) against the false positive rate (FPR) at various thresholds. Higher AUC means better model performance.

We'll implement a vectorized version using NumPy.

### Vectorized Implementation: ROC-AUC

This function sorts the predicted probabilities, computes TPR and FPR at each threshold, and calculates the area under the ROC curve using the trapezoidal rule.

## 6. PR-AUC (Precision-Recall Area Under Curve)

**PR-AUC** measures the area under the precision-recall curve, which is especially useful for imbalanced datasets. Higher PR-AUC means better model performance on the positive class.

We'll implement a vectorized version using NumPy.

In [20]:
def roc_auc_vectorized(true_labels, predicted_probs):
    """Compute ROC-AUC using NumPy vectorization."""
    order = np.argsort(-predicted_probs)
    sorted_true = true_labels[order]
    tps = np.cumsum(sorted_true)
    fps = np.cumsum(1 - sorted_true)
    tpr = tps / np.sum(sorted_true)
    fpr = fps / np.sum(1 - sorted_true)
    auc = np.trapezoid(tpr, fpr)
    return auc, tpr, fpr

### Vectorized Implementation: PR-AUC

This function sorts the predicted probabilities, computes precision and recall at each threshold, and calculates the area under the precision-recall curve using the trapezoidal rule.

In [21]:
def pr_auc_vectorized(true_labels, predicted_probs):
    """Compute PR-AUC using NumPy vectorization."""
    order = np.argsort(-predicted_probs)
    sorted_true = true_labels[order]
    tps = np.cumsum(sorted_true)
    fps = np.cumsum(1 - sorted_true)
    precision = tps / (tps + fps + 1e-8)
    recall = tps / np.sum(sorted_true)
    auc = np.trapezoid(precision, recall)
    return auc, precision, recall

## 7. Log-Loss (Cross-Entropy Loss)

**Log-Loss** measures the performance of a classification model where the output is a probability value between 0 and 1. Lower log-loss indicates better model performance.

**Formula:**
$$\text{LogLoss} = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

We'll implement a vectorized version using NumPy.

### Vectorized Implementation: Log-Loss

This function uses NumPy to efficiently compute log-loss for predicted probabilities.

In [22]:
def log_loss_vectorized(true_labels, predicted_probs):
    """Compute log-loss using NumPy vectorization."""
    eps = 1e-15
    clipped_probs = np.clip(predicted_probs, eps, 1 - eps)
    return -np.mean(true_labels * np.log(clipped_probs) + (1 - true_labels) * np.log(1 - clipped_probs))

## 8. Run All Metrics and Compare Results

Let's run all the above functions on our example data and compare the results from naive and vectorized implementations.

In [23]:


# Example binary classification data
true_labels = np.array([1, 0, 1, 1, 0, 1, 0, 0])
predicted_labels = np.array([1, 0, 1, 0, 0, 1, 0, 1])  # Model's predicted class labels
predicted_probs = np.array([0.9, 0.1, 0.8, 0.4, 0.2, 0.95, 0.3, 0.6])  # Model's predicted probabilities

In [24]:
print("Accuracy (naive):", accuracy_naive(true_labels, predicted_labels))
print("Accuracy (vectorized):", accuracy_vectorized(true_labels, predicted_labels))

print("\nPrecision, Recall, F1 (naive):", precision_recall_f1_naive(true_labels, predicted_labels))
print("Precision, Recall, F1 (vectorized):", precision_recall_f1_vectorized(true_labels, predicted_labels))

print("\nConfusion Matrix (naive):\n", confusion_matrix_naive(true_labels, predicted_labels))
print("Confusion Matrix (vectorized):\n", confusion_matrix_vectorized(true_labels, predicted_labels))

roc_auc, tpr, fpr = roc_auc_vectorized(true_labels, predicted_probs)
print("\nROC-AUC (vectorized):", roc_auc)

pr_auc, precision_curve, recall_curve = pr_auc_vectorized(true_labels, predicted_probs)
print("PR-AUC (vectorized):", pr_auc)

print("\nLog-Loss (vectorized):", log_loss_vectorized(true_labels, predicted_probs))

Accuracy (naive): 0.75
Accuracy (vectorized): 0.75

Precision, Recall, F1 (naive): (0.7499999981250001, 0.7499999981250001, 0.7499999931250001)
Precision, Recall, F1 (vectorized): (np.float64(0.7499999981250001), np.float64(0.7499999981250001), np.float64(0.7499999931250001))

Confusion Matrix (naive):
 [[3 1]
 [1 3]]
Confusion Matrix (vectorized):
 [[3 1]
 [1 3]]

ROC-AUC (vectorized): 0.9375
PR-AUC (vectorized): 0.6937499966489584

Log-Loss (vectorized): 0.36219472950233317
